# FPL Expected Points Model (xPts)
**Structure**
1. Config & Constants
2. Logging Setup
3. API Layer
4. Team Strength (replaces R script)
5. Build Core DataFrames: Fixtures, Performances, Players
6. xPts Model
7. Future Predictions
8. Player Summary
9. Quick Inspection Helpers

---
## Section 1 — Config & Constants

In [4]:
import requests
import numpy as np
import pandas as pd
import logging
import os
import time
from scipy.stats import poisson

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('future.no_silent_downcasting', True)

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_DIR  = '/Users/nathanchbarbi/anaconda_projects/FPL'
BIG5_XG_PATH = os.path.join(PROJECT_DIR, 'big5_xG_data.csv')

# ── FPL API ───────────────────────────────────────────────────────────────────
FPL_BASE  = 'https://fantasy.premierleague.com/api'
API_DELAY = 0.5   # seconds between requests — be polite to the API

# ── Season ────────────────────────────────────────────────────────────────────
CURRENT_SEASON = '2025-26'

# ── Team name mapping: FBRef name → FPL name ─────────────────────────────────
# FBRef uses full names; FPL uses slightly different ones.
# Update this dict if teams are promoted/relegated between seasons.
FBREF_TO_FPL = {
    'Manchester City':  'Man City',
    'Manchester Utd':   'Man Utd',
    'Newcastle Utd':    'Newcastle',
    "Nott'ham Forest":  "Nott'm Forest",
    'Leeds United':     'Leeds',
    'Tottenham':        'Spurs',
}

# ── FPL Scoring Rules ─────────────────────────────────────────────────────────
# Points per event by position.
# play_u60 / play_60plus are cumulative: a player who plays 60+ gets both.
SCORING = {
    'GKP': {
        'play_u60':        1,
        'play_60plus':     2,    # total for 60+ (1 base + 1 bonus)
        'goal':            10,
        'assist':          3,
        'clean_sheet':     4,
        'goals_conceded': -0.5,  # -1 per 2 goals conceded
        'yellow':         -1,
        'red':            -3,
        'save_per_3':      1,    # 1pt per 3 saves
        'pen_save':        5,
        'pen_miss':       -2,
        'own_goal':       -2,
        'defcon_eligible': False,
    },
    'DEF': {
        'play_u60':        1,
        'play_60plus':     2,
        'goal':            6,
        'assist':          3,
        'clean_sheet':     4,
        'goals_conceded': -0.5,
        'yellow':         -1,
        'red':            -3,
        'save_per_3':      0,
        'pen_save':        0,
        'pen_miss':       -2,
        'own_goal':       -2,
        'defcon_eligible': True,
    },
    'MID': {
        'play_u60':        1,
        'play_60plus':     2,
        'goal':            5,
        'assist':          3,
        'clean_sheet':     1,
        'goals_conceded':  0,
        'yellow':         -1,
        'red':            -3,
        'save_per_3':      0,
        'pen_save':        0,
        'pen_miss':       -2,
        'own_goal':       -2,
        'defcon_eligible': True,
    },
    'FWD': {
        'play_u60':        1,
        'play_60plus':     2,
        'goal':            4,
        'assist':          3,
        'clean_sheet':     0,
        'goals_conceded':  0,
        'yellow':         -1,
        'red':            -3,
        'save_per_3':      0,
        'pen_save':        0,
        'pen_miss':       -2,
        'own_goal':       -2,
        'defcon_eligible': True,
    },
}

# Defensive contribution threshold to earn bonus point (total CBI + recoveries + tackles).
# P(defcon bonus) = P(total > threshold) under a Poisson model.
# GKPs are not eligible.
DEFCON_THRESHOLD = {
    'GKP': None,
    'DEF': 10,
    'MID': 12,
    'FWD': 12,
}

# ── Bayesian shrinkage prior ───────────────────────────────────────────────────
# N_PRIOR = equivalent number of prior matches at the positional average.
# Early in the season, rates are pulled toward the position mean.
# As real data accumulates, the player's own numbers dominate.
N_PRIOR = 10

print('Config loaded.')

Config loaded.


---
## Section 2 — Logging Setup

In [6]:
LOG_PATH = os.path.join(PROJECT_DIR, 'fpl_xpts.log')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    handlers=[
        logging.FileHandler(LOG_PATH, mode='w'),  # overwrites log each run
        logging.StreamHandler()                    # also prints to notebook output
    ]
)
log = logging.getLogger(__name__)
log.info('Logging initialised.')

2026-05-14 15:35:07,420  INFO      Logging initialised.


---
## Section 3 — API Layer

All HTTP requests live in this section. The `_fetch` helper handles errors,
retries, and logging in one place — no bare `except` clauses anywhere else.

In [8]:
def _fetch(url: str, retries: int = 3) -> dict | list | None:
    """
    GET a URL and return parsed JSON.

    - Retries on network/timeout errors with exponential backoff.
    - Does NOT retry on HTTP errors (404, 403, etc.) — logs and returns None.
    - Returns None on complete failure so callers can handle gracefully.
    """
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, timeout=10)
            resp.raise_for_status()   # raises HTTPError on 4xx/5xx
            return resp.json()
        except requests.exceptions.HTTPError as e:
            log.error(f'HTTP error fetching {url}: {e}')
            return None              # no point retrying a 404/403
        except requests.exceptions.RequestException as e:
            log.warning(f'Attempt {attempt}/{retries} failed for {url}: {e}')
            if attempt < retries:
                time.sleep(2 ** attempt)  # 2s, 4s backoff
    log.error(f'All {retries} attempts failed for {url}.')
    return None


def fetch_bootstrap() -> dict:
    """Fetch FPL bootstrap-static (teams, players, positions, events)."""
    url  = f'{FPL_BASE}/bootstrap-static/'
    log.info(f'Fetching bootstrap: {url}')
    data = _fetch(url)
    if data is None:
        raise RuntimeError('Failed to fetch bootstrap. Check your internet connection.')
    return data


def fetch_fixtures() -> pd.DataFrame:
    """Fetch all season fixtures. Returns a raw DataFrame."""
    url  = f'{FPL_BASE}/fixtures/'
    log.info(f'Fetching fixtures: {url}')
    data = _fetch(url)
    if data is None:
        raise RuntimeError('Failed to fetch fixtures.')
    return pd.DataFrame(data)


def fetch_element_summary(player_id: int) -> dict | None:
    """
    Fetch per-match history and upcoming fixtures for one player.
    Returns dict with keys 'history' and 'fixtures', or None on failure.
    Applies a short delay to avoid hammering the API.
    """
    url  = f'{FPL_BASE}/element-summary/{player_id}/'
    data = _fetch(url)
    if data is None:
        log.warning(f'Could not fetch element-summary for player {player_id}.')
    time.sleep(API_DELAY)
    return data


def fetch_gw_live(gw: int) -> dict | None:
    """
    Fetch post-match stats for a completed gameweek.
    Returns dict with key 'elements', each containing a 'stats' sub-dict.
    """
    url  = f'{FPL_BASE}/event/{gw}/live/'
    data = _fetch(url)
    if data is None:
        log.warning(f'Could not fetch GW {gw} live data.')
    return data


print('API layer ready.')

API layer ready.


---
## Section 4 — Team Strength (replaces R script)

Reads `big5_xG_data.csv`, computes **log-ratio** strength scores
(chosen over sigmoid/normal for better behaviour in the tails),
and produces one row per PL team with home/away attack and defence scores.

In [10]:
def compute_team_strength(big5_path: str, season: str) -> pd.DataFrame:
    """
    Load big5 xG data, compute log-ratio strength scores, return a
    one-row-per-team DataFrame for the given PL season.

    Log-ratio formula (centred at 50, league average = 50):
        attack_score  = 50 + 50 * log(team_xG  / mean_xG_across_all_data)
        defense_score = 50 + 50 * log(team_xGA / mean_xGA_across_all_data)
            then INVERTED: defense_score = 100 - raw_score
            so that higher = better defence (fewer goals conceded)

    Scores are unbounded but in practice sit between ~20 and ~80.
    50 = exactly league average. Above 50 = better than average.

    Output columns:
        squad, fpl_name,
        home_attack, home_defense, away_attack, away_defense
    """
    df = pd.read_csv(big5_path)

    # Use full cross-league dataset mean as the baseline.
    # This makes scores comparable across seasons even as the league average shifts.
    mean_xG  = df['xG'].mean()
    mean_xGA = df['xGA'].mean()

    df['attack_score']  = 50 + 50 * np.log(df['xG']  / mean_xG)
    df['defense_score'] = 100 - (50 + 50 * np.log(df['xGA'] / mean_xGA))
    #                     ^--- invert: higher xGA = weaker defence, we flip it

    # Filter to PL current season
    pl = df[(df['League'] == 'Premier League') & (df['Season'] == season)].copy()
    if pl.empty:
        available = df['Season'].unique().tolist()
        raise ValueError(
            f'No PL data for season "{season}". Available: {available}'
        )

    # Extract Home / Away from team_id (format: "Arsenal_Home_202526")
    pl['home_away'] = pl['team_id'].str.extract(r'_(Home|Away)_')

    # Pivot: one row per team, separate home/away columns
    pl_pivot = pl.pivot_table(
        index='Squad',
        columns='home_away',
        values=['attack_score', 'defense_score']
    )
    pl_pivot.columns = [
        f'{ha.lower()}_{metric.replace("_score", "")}'
        for metric, ha in pl_pivot.columns
    ]
    pl_pivot = pl_pivot.round(2).reset_index()
    pl_pivot.rename(columns={'Squad': 'squad'}, inplace=True)

    # Map FBRef → FPL team names
    pl_pivot['fpl_name'] = pl_pivot['squad'].replace(FBREF_TO_FPL)

    log.info(f'Team strength computed: {len(pl_pivot)} PL teams for {season}.')
    return pl_pivot


team_strength = compute_team_strength(BIG5_XG_PATH, CURRENT_SEASON)

print(f'Team strength shape: {team_strength.shape}')
team_strength.sort_values('home_attack', ascending=False)

2026-05-14 15:35:07,461  INFO      Team strength computed: 20 PL teams for 2025-26.


Team strength shape: (20, 6)


,squad,away_attack,home_attack,away_defense,home_defense,fpl_name
12,Manchester City,59.54,73.31,52.85,68.29,Man City
13,Manchester Utd,59.23,71.90,38.37,67.76,Man Utd
7,Crystal Palace,51.27,66.89,49.46,60.43,Crystal Palace
14,Newcastle Utd,23.64,65.83,67.76,56.50,Newcastle
10,Leeds United,35.30,63.07,34.98,64.21,Leeds
15,Nott'ham Forest,16.97,62.50,47.31,48.02,Nott'm Forest
0,Arsenal,59.23,62.21,78.27,116.28,Arsenal
3,Brentford,39.11,62.21,51.70,52.08,Brentford
6,Chelsea,58.61,61.33,48.02,57.78,Chelsea
4,Brighton,52.69,60.74,53.64,49.09,Brighton


---
## Section 5 — Build Core DataFrames

### 5a — Bootstrap: Teams, Positions, Players (raw)

In [12]:
bootstrap = fetch_bootstrap()

# ── Teams ─────────────────────────────────────────────────────────────────────
teams_raw = pd.DataFrame(bootstrap['teams'])
team_id_to_name = dict(zip(teams_raw['id'], teams_raw['name']))
team_name_to_id = dict(zip(teams_raw['name'], teams_raw['id']))

# ── Positions ─────────────────────────────────────────────────────────────────
positions_raw = pd.DataFrame(bootstrap['element_types'])
pos_map = dict(zip(positions_raw['id'], positions_raw['singular_name_short']))

# ── Players (raw from bootstrap) ──────────────────────────────────────────────
players_raw = pd.DataFrame(bootstrap['elements'])
players_raw = players_raw[
    (players_raw['can_select'] == True) &
    (players_raw['minutes'] > 0)
].copy()

players_raw['position']  = players_raw['element_type'].map(pos_map)
players_raw['team_name'] = players_raw['team'].map(team_id_to_name)
players_raw['price_m']   = players_raw['now_cost'] / 10

log.info(f'Bootstrap: {len(players_raw)} eligible players, {len(teams_raw)} teams.')
print(f'Players: {len(players_raw)} | Teams: {len(teams_raw)}')

2026-05-14 15:35:07,479  INFO      Fetching bootstrap: https://fantasy.premierleague.com/api/bootstrap-static/
2026-05-14 15:35:07,736  INFO      Bootstrap: 490 eligible players, 20 teams.


Players: 490 | Teams: 20


### 5b — Fixtures DataFrame

In [14]:
def build_fixtures_df() -> tuple[pd.DataFrame, int]:
    """
    Build a clean Fixtures DataFrame and determine the next gameweek.

    Double gameweeks are handled naturally: a team appears twice in one GW.
    Blank gameweeks are handled naturally: a team simply has no row that GW.
    Both are detected from the data — nothing is hardcoded.

    Returns:
        fixtures  — one row per fixture
        next_gw   — integer GW number of the next unplayed gameweek
    """
    raw = fetch_fixtures()

    raw['home_team'] = raw['team_h'].map(team_id_to_name)
    raw['away_team'] = raw['team_a'].map(team_id_to_name)
    raw.rename(columns={
        'team_h_score': 'home_score',
        'team_a_score': 'away_score'
    }, inplace=True)

    keep = [
        'id', 'event', 'finished',
        'team_h', 'team_a', 'home_team', 'away_team',
        'home_score', 'away_score',
        'team_h_difficulty', 'team_a_difficulty',
        'kickoff_time'
    ]
    fixtures = raw[keep].dropna(subset=['event']).copy()
    fixtures['event'] = fixtures['event'].astype(int)

    unplayed = fixtures[fixtures['finished'] == False]
    next_gw  = int(unplayed['event'].min()) if not unplayed.empty else int(fixtures['event'].max()) + 1

    log.info(f'Fixtures: {len(fixtures)} total, {fixtures["finished"].sum()} completed. Next GW: {next_gw}.')
    return fixtures, next_gw


fixtures, next_gw = build_fixtures_df()

print(f'Next gameweek: GW{next_gw}')
fixtures.head()

2026-05-14 15:35:07,750  INFO      Fetching fixtures: https://fantasy.premierleague.com/api/fixtures/
2026-05-14 15:35:07,909  INFO      Fixtures: 380 total, 360 completed. Next GW: 37.


Next gameweek: GW37


,id,event,finished,team_h,team_a,home_team,away_team,home_score,away_score,team_h_difficulty,team_a_difficulty,kickoff_time
0,1,1,True,12,4,Liverpool,Bournemouth,4.0,2.0,3,4,2025-08-15T19:00:00Z
1,2,1,True,2,15,Aston Villa,Newcastle,0.0,0.0,3,4,2025-08-16T11:30:00Z
2,3,1,True,6,10,Brighton,Fulham,1.0,1.0,3,4,2025-08-16T14:00:00Z
3,6,1,True,18,3,Spurs,Burnley,3.0,0.0,1,3,2025-08-16T14:00:00Z
4,5,1,True,17,19,Sunderland,West Ham,3.0,0.0,2,3,2025-08-16T14:00:00Z


### 5c — Performances DataFrame

Two data sources merged on `(player_id, gameweek)`:

| Source | Contains |
|---|---|
| `element-summary/{id}/` | xG, xA, xGC, BPS, ICT, goals, assists, saves, minutes, per-fixture detail |
| `event/{gw}/live/` | CBI, recoveries, tackles (defensive stats only in this endpoint) |

> ⚠️ This cell makes ~600 API calls (one per player). At 0.5s delay it takes ~5 minutes. **Run once and save.**

In [16]:
def build_performances_df(
    players_raw: pd.DataFrame,
    fixtures: pd.DataFrame,
    next_gw: int
) -> pd.DataFrame:
    """
    Build the Performances DataFrame — one row per (player, fixture) for all
    completed gameweeks.

    Step 1: element-summary per player  → rich per-fixture stats
    Step 2: event/live per GW           → CBI, recoveries, tackles
    Step 3: merge on (player_id, gameweek), enrich with metadata
    """

    # ── Step 1: element-summary ───────────────────────────────────────────────
    log.info('Fetching element-summary for all players...')
    summary_rows = []
    player_ids   = players_raw['id'].tolist()

    for i, pid in enumerate(player_ids):
        if i % 50 == 0:
            log.info(f'  element-summary: {i}/{len(player_ids)}')

        data = fetch_element_summary(pid)
        if not data or not data.get('history'):
            continue

        for m in data['history']:
            summary_rows.append({
                'player_id':        pid,
                'fixture_id':       m.get('fixture'),
                'gameweek':         m.get('round'),
                'opponent_team_id': m.get('opponent_team'),
                'was_home':         m.get('was_home'),
                'minutes':          m.get('minutes', 0),
                'goals_scored':     m.get('goals_scored', 0),
                'assists':          m.get('assists', 0),
                'clean_sheets':     m.get('clean_sheets', 0),
                'goals_conceded':   m.get('goals_conceded', 0),
                'own_goals':        m.get('own_goals', 0),
                'penalties_saved':  m.get('penalties_saved', 0),
                'penalties_missed': m.get('penalties_missed', 0),
                'yellow_cards':     m.get('yellow_cards', 0),
                'red_cards':        m.get('red_cards', 0),
                'saves':            m.get('saves', 0),
                'bonus':            m.get('bonus', 0),
                'bps':              m.get('bps', 0),
                'influence':        float(m.get('influence', 0)),
                'creativity':       float(m.get('creativity', 0)),
                'threat':           float(m.get('threat', 0)),
                'ict_index':        float(m.get('ict_index', 0)),
                'xG':               float(m.get('expected_goals', 0)),
                'xA':               float(m.get('expected_assists', 0)),
                'xGC':              float(m.get('expected_goals_conceded', 0)),
                'xGI':              float(m.get('expected_goal_involvements', 0)),
                'total_points':     m.get('total_points', 0),
            })

    df_summary = pd.DataFrame(summary_rows)
    log.info(f'element-summary: {len(df_summary)} player-fixture rows.')

    # ── Step 2: event/live per GW ─────────────────────────────────────────────
    completed_gws = sorted(fixtures[fixtures['finished'] == True]['event'].unique())
    log.info(f'Fetching GW live data for {len(completed_gws)} completed GWs...')

    live_rows = []
    for gw in completed_gws:
        data = fetch_gw_live(gw)
        if not data:
            log.warning(f'Skipping GW {gw} — no live data returned.')
            continue
        for elem in data['elements']:
            s = elem.get('stats', {})
            live_rows.append({
                'player_id':              elem['id'],
                'gameweek':               gw,
                'cbi':                    s.get('clearances_blocks_interceptions', 0),
                'recoveries':             s.get('recoveries', 0),
                'tackles':                s.get('tackles', 0),
                'defensive_contribution': s.get('defensive_contribution', 0),
            })

    df_live = pd.DataFrame(live_rows)
    log.info(f'GW live: {len(df_live)} player-GW rows.')

    # ── Step 3: Merge ─────────────────────────────────────────────────────────
    # Note on double GWs: element-summary has 2 fixture rows per GW;
    # live has 1 GW-aggregated row. Defensive stats are assigned to
    # both fixtures via the left merge — acceptable for now.
    df = df_summary.merge(df_live, on=['player_id', 'gameweek'], how='left')

    # Enrich with player metadata
    meta = players_raw[[
        'id', 'first_name', 'second_name', 'web_name', 'position', 'team_name', 'team'
    ]].copy()
    df = df.merge(meta, left_on='player_id', right_on='id', how='left')
    df['opponent_team'] = df['opponent_team_id'].map(team_id_to_name)

    # Remove non-appearances
    df = df[df['minutes'] > 0].copy()

    # Fill defensive NaNs with 0 (players with no live data for that GW)
    for col in ['cbi', 'recoveries', 'tackles', 'defensive_contribution']:
        df[col] = df[col].fillna(0).astype(int)

    # Final column order
    col_order = [
        'player_id', 'first_name', 'second_name', 'web_name',
        'position', 'team_name', 'gameweek',
        'fixture_id', 'opponent_team', 'was_home',
        'minutes', 'goals_scored', 'assists', 'clean_sheets',
        'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed',
        'yellow_cards', 'red_cards', 'saves',
        'cbi', 'recoveries', 'tackles', 'defensive_contribution',
        'bonus', 'bps', 'influence', 'creativity', 'threat', 'ict_index',
        'xG', 'xA', 'xGC', 'xGI',
        'total_points'
    ]
    df = df[col_order].reset_index(drop=True)

    log.info(f'Performances built: {df.shape}')
    return df


performances = build_performances_df(players_raw, fixtures, next_gw)

# Save — avoids re-fetching on subsequent runs
performances.to_csv(os.path.join(PROJECT_DIR, 'performances.csv'), index=False)
log.info('Performances saved.')

print(f'Performances: {performances.shape}')
performances.head()

2026-05-14 15:35:07,938  INFO      Fetching element-summary for all players...
2026-05-14 15:35:07,938  INFO        element-summary: 0/490
2026-05-14 15:35:38,958  INFO        element-summary: 50/490
2026-05-14 15:36:09,897  INFO        element-summary: 100/490
2026-05-14 15:36:41,140  INFO        element-summary: 150/490
2026-05-14 15:37:12,356  INFO        element-summary: 200/490
2026-05-14 15:37:43,584  INFO        element-summary: 250/490
2026-05-14 15:38:14,853  INFO        element-summary: 300/490
2026-05-14 15:38:45,835  INFO        element-summary: 350/490
2026-05-14 15:39:17,961  INFO        element-summary: 400/490
2026-05-14 15:39:49,151  INFO        element-summary: 450/490
2026-05-14 15:40:14,890  INFO      element-summary: 16940 player-fixture rows.
2026-05-14 15:40:14,902  INFO      Fetching GW live data for 36 completed GWs...
2026-05-14 15:40:19,235  INFO      GW live: 28066 player-GW rows.
2026-05-14 15:40:19,281  INFO      Performances built: (10620, 36)
2026-05-14 

Performances: (10620, 36)


,player_id,first_name,second_name,web_name,position,team_name,gameweek,fixture_id,opponent_team,was_home,minutes,goals_scored,assists,clean_sheets,goals_conceded,own_goals,penalties_saved,penalties_missed,yellow_cards,red_cards,saves,cbi,recoveries,tackles,defensive_contribution,bonus,bps,influence,creativity,threat,ict_index,xG,xA,xGC,xGI,total_points
0,1,David,Raya Martín,Raya,GKP,Arsenal,1,9,Man Utd,False,90,0,0,1,0,0,0,0,1,0,7,1,13,0,0,3,38,49.2,0.0,0.0,4.9,0.0,0.00,1.52,0.00,10
1,1,David,Raya Martín,Raya,GKP,Arsenal,2,11,Leeds,True,90,0,0,1,0,0,0,0,0,0,1,0,3,0,0,0,28,13.4,0.0,0.0,1.3,0.0,0.00,0.17,0.00,6
2,1,David,Raya Martín,Raya,GKP,Arsenal,3,25,Liverpool,False,90,0,0,0,1,0,0,0,0,0,2,0,12,0,0,0,12,20.0,10.0,0.0,3.0,0.0,0.02,0.52,0.02,2
3,1,David,Raya Martín,Raya,GKP,Arsenal,4,31,Nott'm Forest,True,90,0,0,1,0,0,0,0,0,0,1,0,9,0,0,0,24,12.8,0.0,0.0,1.3,0.0,0.00,0.20,0.00,6
4,1,David,Raya Martín,Raya,GKP,Arsenal,5,41,Man City,True,90,0,0,0,1,0,0,0,0,0,2,1,6,0,0,0,13,21.4,0.0,0.0,2.1,0.0,0.01,0.89,0.01,2


---
## Section 6 — xPts Model

### 6a — Compute per-match Poisson rates (λ) with Bayesian shrinkage

In [18]:
def compute_player_rates(
    performances: pd.DataFrame,
    players_raw: pd.DataFrame,
    n_prior: int = N_PRIOR
) -> pd.DataFrame:
    """
    Compute Bayesian-shrunk Poisson rate (λ) for each player for each stat.

    Formula:
        λ_player = (player_season_total + n_prior * position_avg_rate)
                   / (player_MP + n_prior)

    Early in the season (small MP), rates lean on the positional average.
    Later, the player's own data dominates.

    Returns a DataFrame indexed by player_id.
    """
    rate_cols = [
        'goals_scored', 'assists', 'clean_sheets', 'goals_conceded',
        'saves', 'yellow_cards', 'red_cards',
        'penalties_saved', 'penalties_missed',
        'cbi', 'recoveries', 'tackles', 'defensive_contribution',
        'xG', 'xA', 'xGC', 'minutes'
    ]

    # Season totals
    totals      = performances.groupby('player_id')[rate_cols].sum()
    totals['MP'] = performances[performances['minutes'] > 0].groupby('player_id').size()

    # Attach position
    pos_lookup    = players_raw.set_index('id')['position']
    totals['position'] = totals.index.map(pos_lookup)

    # Positional averages (prior)
    pos_totals = totals.groupby('position')[rate_cols].sum()
    pos_mp     = totals.groupby('position')['MP'].sum()
    pos_rate   = pos_totals.div(pos_mp, axis=0)   # per-match rate per position

    # Build rates DataFrame
    rates = pd.DataFrame(index=totals.index)
    rates['position']    = totals['position']
    rates['MP']          = totals['MP']
    rates['avg_minutes'] = (totals['minutes'] / totals['MP']).round(1)

    for col in rate_cols:
        if col == 'minutes':
            continue
        prior          = totals['position'].map(pos_rate[col])
        rates[f'lam_{col}'] = (
            (totals[col] + n_prior * prior) / (totals['MP'] + n_prior)
        ).round(4)

    log.info(f'Player rates computed for {len(rates)} players.')
    return rates


player_rates = compute_player_rates(performances, players_raw)
print(f'Player rates: {player_rates.shape}')
player_rates.head()

2026-05-14 15:40:19,442  INFO      Player rates computed for 490 players.


Player rates: (490, 19)


,position,MP,avg_minutes,lam_goals_scored,lam_assists,lam_clean_sheets,lam_goals_conceded,lam_saves,lam_yellow_cards,lam_red_cards,lam_penalties_saved,lam_penalties_missed,lam_cbi,lam_recoveries,lam_tackles,lam_defensive_contribution,lam_xG,lam_xA,lam_xGC
player_id,,,,,,,,,,,,,,,,,,,
1,GKP,36,90.0,0.0000,0.0012,0.4472,0.8616,1.8994,0.0373,0.0003,0.0033,0.0,1.1429,8.6688,0.0292,0.0000,0.0000,0.0017,0.8937
5,DEF,30,87.2,0.0837,0.1402,0.4787,0.7536,0.0000,0.1369,0.0016,0.0000,0.0,7.5178,2.5387,1.3804,8.8982,0.0848,0.0543,0.8031
6,DEF,30,84.1,0.0337,0.0152,0.4037,0.7786,0.0000,0.0869,0.0016,0.0000,0.0,5.0178,3.5387,1.2304,6.2482,0.0333,0.0416,0.7863
7,DEF,24,65.9,0.0396,0.0768,0.4161,0.5630,0.0000,0.1904,0.0019,0.0000,0.0,3.5209,2.9867,1.3593,4.8802,0.1121,0.0292,0.6092
8,DEF,30,81.7,0.0837,0.1652,0.3787,0.7786,0.0000,0.1619,0.0016,0.0000,0.0,3.3928,3.4387,2.0804,5.4732,0.1291,0.0498,0.7171


### 6b — Opponent Strength Adjustment

Uses the log-ratio team strength scores directly as multipliers:

```
att_multiplier = (100 - opp_defense) / 50   # weaker opp defence  → > 1
def_multiplier = opp_attack / 50             # stronger opp attack → > 1
```

Centred at 1.0: an average opponent produces no adjustment.

In [20]:
def build_team_strength_lookup(
    team_strength: pd.DataFrame,
    teams_raw: pd.DataFrame
) -> pd.DataFrame:
    """
    Merge team strength scores onto FPL teams, indexed by FPL team id.
    Warns if any team is missing from the strength data.
    """
    teams = teams_raw[['id', 'name']].copy()
    teams = teams.merge(
        team_strength[['fpl_name', 'home_attack', 'home_defense', 'away_attack', 'away_defense']],
        left_on='name', right_on='fpl_name', how='left'
    )
    missing = teams[teams['home_attack'].isna()]['name'].tolist()
    if missing:
        log.warning(
            f'No strength data for: {missing}. '
            f'Add entries to FBREF_TO_FPL or check big5_xG_data.csv.'
        )
    return teams.set_index('id')


def adjust_rates_for_opponent(
    rates_row: pd.Series,
    opp_team_id: int,
    is_home: bool,
    team_lookup: pd.DataFrame
) -> pd.Series:
    """
    Scale a player's Poisson rates for a specific opponent and home/away context.

    When the player is at home, the opponent uses their away ratings.
    When the player is away, the opponent uses their home ratings.

    Attacking rates (goals, assists, xG, xA) scaled by:
        (100 - opp_defense) / 50    ← weaker opp defence = easier to score

    Defensive rates (goals_conceded, xGC) scaled by:
        opp_attack / 50             ← stronger opp attack = more goals conceded
    """
    row = rates_row.copy()

    if opp_team_id not in team_lookup.index:
        log.warning(f'opp_team_id {opp_team_id} not in team_lookup — no adjustment applied.')
        return row

    opp = team_lookup.loc[opp_team_id]

    # Opponent uses away context when player is home, and vice versa
    opp_att = opp['away_attack']   if is_home else opp['home_attack']
    opp_def = opp['away_defense']  if is_home else opp['home_defense']

    att_mult = (100 - opp_def) / 50
    def_mult = opp_att / 50

    for col in ['lam_goals_scored', 'lam_assists', 'lam_xG', 'lam_xA']:
        if col in row.index:
            row[col] = max(0.0, float(row[col]) * att_mult)

    for col in ['lam_goals_conceded', 'lam_xGC']:
        if col in row.index:
            row[col] = max(0.0, float(row[col]) * def_mult)

    return row


team_lookup = build_team_strength_lookup(team_strength, teams_raw)
print('Team lookup ready.')
team_lookup[['name', 'home_attack', 'home_defense', 'away_attack', 'away_defense']].sort_values('home_attack', ascending=False)

Team lookup ready.


,name,home_attack,home_defense,away_attack,away_defense
id,,,,,
13,Man City,73.31,68.29,59.54,52.85
14,Man Utd,71.90,67.76,59.23,38.37
8,Crystal Palace,66.89,60.43,51.27,49.46
15,Newcastle,65.83,56.50,23.64,67.76
11,Leeds,63.07,64.21,35.30,34.98
16,Nott'm Forest,62.50,48.02,16.97,47.31
1,Arsenal,62.21,116.28,59.23,78.27
5,Brentford,62.21,52.08,39.11,51.70
7,Chelsea,61.33,57.78,58.61,48.02


### 6c — Unified xPts Function

Single function for both historical and projected xPts.
- `mode='projected'` — uses Poisson rates (λ) for probabilistic modelling
- `mode='historic'`  — uses actual match stats + Opta xG/xGC where appropriate

In [22]:
def _ev_poisson(lam: float, pts_per: float, k_max: int = 10) -> float:
    """Expected points from a Poisson-distributed event. E[pts] = sum(k * pts_per * P(X=k))."""
    if lam <= 0:
        return 0.0
    return sum(k * pts_per * poisson.pmf(k, mu=lam) for k in range(1, k_max + 1))


def compute_xpts(row: pd.Series, mode: str = 'projected') -> dict:
    """
    Compute xPts and all component contributions for one player-fixture.

    Parameters
    ----------
    row  : pd.Series — either a player_rates row (projected) or performances row (historic)
    mode : 'projected' | 'historic'

    Returns
    -------
    dict with keys: xPts, xPts_mins, xPts_cs, xPts_gc, xPts_goals, xPts_assists,
                    xPts_saves, xPts_yellows, xPts_reds, xPts_pen_save,
                    xPts_pen_miss, xPts_defcon
    """
    pos = row.get('position')
    if pos not in SCORING:
        return {'xPts': 0.0}

    s   = SCORING[pos]
    out = {}

    # ── Playing time ──────────────────────────────────────────────────────────
    if mode == 'projected':
        avg_mins     = float(row.get('avg_minutes', 60))
        prob_plays60 = 1.0 - poisson.cdf(59, mu=avg_mins)
        # 1pt for playing at all (assumed starter), +1pt if 60+ mins
        out['xPts_mins'] = round(
            s['play_u60'] + (s['play_60plus'] - s['play_u60']) * prob_plays60, 3
        )
    else:
        mins = float(row.get('minutes', 0))
        out['xPts_mins'] = (
            s['play_u60'] * int(mins > 0) +
            (s['play_60plus'] - s['play_u60']) * int(mins >= 60)
        )

    # ── Clean sheet ───────────────────────────────────────────────────────────
    # P(CS) = P(goals_conceded = 0) = exp(-λ_gc)
    # Conditional on playing 60+ minutes (necessary condition for CS award).
    if mode == 'projected':
        lam_gc  = float(row.get('lam_goals_conceded', 0))
        prob_cs = np.exp(-lam_gc) * prob_plays60
    else:
        # Use Opta xGC as the best estimate of expected goals conceded in that match
        lam_gc  = float(row.get('xGC', 0))
        prob_cs = np.exp(-lam_gc) * int(float(row.get('minutes', 0)) >= 60)
    out['xPts_cs'] = round(s['clean_sheet'] * prob_cs, 3)

    # ── Goals conceded (GKP/DEF penalty) ─────────────────────────────────────
    if s['goals_conceded'] != 0:
        gc_lam = float(row.get('lam_goals_conceded' if mode == 'projected' else 'goals_conceded', 0))
        # E[goals conceded] = λ; -0.5 pts per goal (-1 per 2)
        out['xPts_gc'] = round(s['goals_conceded'] * gc_lam, 3)
    else:
        out['xPts_gc'] = 0.0

    # ── Goals scored ──────────────────────────────────────────────────────────
    # Projected: use shrunk goal-scoring rate
    # Historic:  use Opta xG as λ (better reflection of quality of chances)
    lam_g          = float(row.get('lam_goals_scored' if mode == 'projected' else 'xG', 0))
    out['xPts_goals'] = round(_ev_poisson(lam_g, s['goal'], k_max=5), 3)

    # ── Assists ───────────────────────────────────────────────────────────────
    lam_a             = float(row.get('lam_assists' if mode == 'projected' else 'xA', 0))
    out['xPts_assists'] = round(_ev_poisson(lam_a, s['assist'], k_max=5), 3)

    # ── Saves (GKP only) ──────────────────────────────────────────────────────
    # FPL: 1pt per 3 saves = 1/3 pt per save
    if s['save_per_3'] > 0:
        lam_sv         = float(row.get('lam_saves' if mode == 'projected' else 'saves', 0))
        out['xPts_saves'] = round(_ev_poisson(lam_sv, 1 / 3, k_max=15), 3)
    else:
        out['xPts_saves'] = 0.0

    # ── Yellow cards ──────────────────────────────────────────────────────────
    # P(exactly 1 yellow) — 2-yellow reds appear in red_cards, not here.
    lam_y              = float(row.get('lam_yellow_cards' if mode == 'projected' else 'yellow_cards', 0))
    out['xPts_yellows'] = round(s['yellow'] * poisson.pmf(1, mu=lam_y), 3)

    # ── Red cards ─────────────────────────────────────────────────────────────
    # Captures both straight reds and 2-yellow reds (both recorded as red_cards).
    lam_r            = float(row.get('lam_red_cards' if mode == 'projected' else 'red_cards', 0))
    out['xPts_reds'] = round(s['red'] * poisson.pmf(1, mu=lam_r), 3)

    # ── Penalty saves ─────────────────────────────────────────────────────────
    if s['pen_save'] > 0:
        lam_ps              = float(row.get('lam_penalties_saved' if mode == 'projected' else 'penalties_saved', 0))
        out['xPts_pen_save'] = round(_ev_poisson(lam_ps, s['pen_save'], k_max=3), 3)
    else:
        out['xPts_pen_save'] = 0.0

    # ── Penalty misses ────────────────────────────────────────────────────────
    lam_pm              = float(row.get('lam_penalties_missed' if mode == 'projected' else 'penalties_missed', 0))
    out['xPts_pen_miss'] = round(_ev_poisson(lam_pm, s['pen_miss'], k_max=3), 3)

    # ── Defensive contribution bonus ──────────────────────────────────────────
    # 2 bonus pts if total defensive actions exceed position-specific threshold.
    # GKPs are explicitly excluded.
    if s['defcon_eligible']:
        threshold       = DEFCON_THRESHOLD.get(pos, 999)
        lam_dc          = float(row.get(
            'lam_defensive_contribution' if mode == 'projected' else 'defensive_contribution', 0
        ))
        prob_defcon     = 1.0 - poisson.cdf(threshold - 1, mu=lam_dc)
        out['xPts_defcon'] = round(2.0 * prob_defcon, 3)
    else:
        out['xPts_defcon'] = 0.0

    # ── Total ─────────────────────────────────────────────────────────────────
    component_keys = [
        'xPts_mins', 'xPts_cs', 'xPts_gc', 'xPts_goals', 'xPts_assists',
        'xPts_saves', 'xPts_yellows', 'xPts_reds',
        'xPts_pen_save', 'xPts_pen_miss', 'xPts_defcon'
    ]
    out['xPts'] = round(sum(out[k] for k in component_keys), 2)
    return out


print('xPts function ready.')

xPts function ready.


### 6d — Apply Historical xPts to Performances

In [24]:
# Sort chronologically before rolling calculations
performances = performances.sort_values(['player_id', 'gameweek']).reset_index(drop=True)

# Compute historic xPts for each performance row
xpts_hist = performances.apply(lambda r: compute_xpts(r, mode='historic'), axis=1)
performances = pd.concat([performances, pd.DataFrame(xpts_hist.tolist(), index=performances.index)], axis=1)

# Rolling form metrics
for window, col in [(5, 'xForm5'), (10, 'xForm10')]:
    performances[col] = (
        performances.groupby('player_id')['xPts']
        .transform(lambda x: x.rolling(window, min_periods=1).mean().round(2))
    )

# Save updated performances
performances.to_csv(os.path.join(PROJECT_DIR, 'performances.csv'), index=False)

print(f'Performances with xPts: {performances.shape}')
performances[['web_name', 'gameweek', 'xPts', 'xForm5', 'total_points']].head(20)

Performances with xPts: (10620, 50)


,web_name,gameweek,xPts,xForm5,total_points
0,Raya,1,4.83,4.83,10
1,Raya,2,5.71,5.27,6
2,Raya,3,4.61,5.05,2
3,Raya,4,5.61,5.19,6
4,Raya,5,3.84,4.92,2
5,Raya,6,4.37,4.83,2
6,Raya,7,4.45,4.58,6
7,Raya,8,4.58,4.57,6
8,Raya,9,4.83,4.41,6
9,Raya,10,4.63,4.57,6


---
## Section 7 — Future Predictions DataFrame

One row per **(player, fixture)** for the next N gameweeks.

- **Double GW**: player has 2 rows in the same GW — xPts summed in Player Summary
- **Blank GW**: player has no rows — xPts = 0 in Player Summary
- **Availability**: next GW xPts scaled by `chance_of_playing_next_round / 100`

In [26]:
def build_future_predictions(
    player_rates: pd.DataFrame,
    players_raw: pd.DataFrame,
    fixtures: pd.DataFrame,
    team_lookup: pd.DataFrame,
    next_gw: int,
    n_gws: int = 5
) -> pd.DataFrame:
    """
    Build the Future Predictions DataFrame.
    One row per (player, fixture) for the next n_gws gameweeks.
    """
    gw_range      = range(next_gw, next_gw + n_gws)
    future_fx     = fixtures[
        fixtures['event'].isin(gw_range) &
        (fixtures['finished'] == False)
    ].copy()

    # Long table: one row per (team, fixture)
    fx_home = future_fx[['id', 'event', 'team_h', 'team_a']].copy()
    fx_home.rename(columns={'team_h': 'team_id', 'team_a': 'opp_team_id'}, inplace=True)
    fx_home['is_home'] = True

    fx_away = future_fx[['id', 'event', 'team_h', 'team_a']].copy()
    fx_away.rename(columns={'team_a': 'team_id', 'team_h': 'opp_team_id'}, inplace=True)
    fx_away['is_home'] = False

    fx_long = pd.concat([fx_home, fx_away], ignore_index=True)
    fx_long.rename(columns={'id': 'fixture_id', 'event': 'gameweek'}, inplace=True)

    # Player metadata
    player_meta = players_raw[[
        'id', 'web_name', 'first_name', 'second_name',
        'position', 'team', 'team_name', 'price_m',
        'chance_of_playing_next_round'
    ]].copy()
    player_meta['chance_of_playing_next_round'] = (
        player_meta['chance_of_playing_next_round'].fillna(100.0)
    )

    # Each player gets a row for every fixture their team plays
    pred_df = player_meta.merge(fx_long, left_on='team', right_on='team_id', how='inner')

    # Attach player rates
    pred_df = pred_df.merge(
        player_rates.drop(columns=['position']),
        left_on='id', right_index=True, how='left'
    )
    pred_df['opponent_team'] = pred_df['opp_team_id'].map(team_id_to_name)

    # Compute opponent-adjusted xPts per fixture
    xpts_results = []
    for _, row in pred_df.iterrows():
        adj_row = adjust_rates_for_opponent(
            row,
            opp_team_id=int(row['opp_team_id']),
            is_home=bool(row['is_home']),
            team_lookup=team_lookup
        )
        xpts_results.append(compute_xpts(adj_row, mode='projected'))

    xpts_df = pd.DataFrame(xpts_results, index=pred_df.index)
    pred_df = pd.concat([pred_df, xpts_df], axis=1)

    # Scale next GW xPts by player availability
    is_next = pred_df['gameweek'] == next_gw
    avail   = pred_df['chance_of_playing_next_round'] / 100.0
    pred_df.loc[is_next, 'xPts'] = (pred_df.loc[is_next, 'xPts'] * avail[is_next]).round(2)

    # Column selection
    out_cols = [
        'id', 'web_name', 'first_name', 'second_name', 'position',
        'team_name', 'price_m', 'gameweek', 'fixture_id',
        'opponent_team', 'is_home',
        'avg_minutes', 'MP', 'chance_of_playing_next_round',
        'xPts',
        'xPts_mins', 'xPts_cs', 'xPts_gc', 'xPts_goals', 'xPts_assists',
        'xPts_saves', 'xPts_yellows', 'xPts_reds',
        'xPts_pen_save', 'xPts_pen_miss', 'xPts_defcon',
        'lam_goals_scored', 'lam_assists', 'lam_goals_conceded',
        'lam_saves', 'lam_yellow_cards', 'lam_red_cards',
        'lam_xG', 'lam_xA', 'lam_xGC',
    ]
    pred_df = pred_df[[c for c in out_cols if c in pred_df.columns]]
    pred_df = pred_df.sort_values(['id', 'gameweek', 'fixture_id']).reset_index(drop=True)

    log.info(f'Future predictions built: {pred_df.shape}')
    return pred_df


future_predictions = build_future_predictions(
    player_rates, players_raw, fixtures, team_lookup, next_gw, n_gws=5
)

future_predictions.to_csv(
    os.path.join(PROJECT_DIR, 'future_predictions.csv'), index=False
)

# Spot-check: any double GWs?
dgw_check = future_predictions.groupby(['id', 'gameweek']).size()
dgw_players = dgw_check[dgw_check > 1]
print(f'Future predictions: {future_predictions.shape}')
print(f'Double GW fixtures found: {len(dgw_players)} player-GW combinations')
if not dgw_players.empty:
    print(dgw_players.head(10))
future_predictions.head(10)

2026-05-14 15:40:23,416  INFO      Future predictions built: (980, 35)


Future predictions: (980, 35)
Double GW fixtures found: 0 player-GW combinations


,id,web_name,first_name,second_name,position,team_name,price_m,gameweek,fixture_id,opponent_team,is_home,avg_minutes,MP,chance_of_playing_next_round,xPts,xPts_mins,xPts_cs,xPts_gc,xPts_goals,xPts_assists,xPts_saves,xPts_yellows,xPts_reds,xPts_pen_save,xPts_pen_miss,xPts_defcon,lam_goals_scored,lam_assists,lam_goals_conceded,lam_saves,lam_yellow_cards,lam_red_cards,lam_xG,lam_xA,lam_xGC
0,1,Raya,David,Raya Martín,GKP,Arsenal,6.1,37,361,Burnley,True,90.0,36,100.0,4.74,2.000,2.381,-0.259,0.000,0.005,0.633,-0.036,-0.001,0.016,0.0,0.000,0.0000,0.0012,0.8616,1.8994,0.0373,0.0003,0.0000,0.0017,0.8937
1,1,Raya,David,Raya Martín,GKP,Arsenal,6.1,38,373,Crystal Palace,False,90.0,36,100.0,3.30,2.000,1.263,-0.576,0.000,0.003,0.633,-0.036,-0.001,0.016,0.0,0.000,0.0000,0.0012,0.8616,1.8994,0.0373,0.0003,0.0000,0.0017,0.8937
2,5,Gabriel,Gabriel,dos Santos Magalhães,DEF,Arsenal,7.3,37,361,Burnley,True,87.2,30,100.0,6.38,1.999,2.539,-0.227,0.761,0.638,0.000,-0.119,-0.005,0.000,0.0,0.798,0.0837,0.1402,0.7536,0.0000,0.1369,0.0016,0.0848,0.0543,0.8031
3,5,Gabriel,Gabriel,dos Santos Magalhães,DEF,Arsenal,7.3,38,373,Crystal Palace,False,87.2,30,100.0,4.36,1.999,1.458,-0.504,0.397,0.333,0.000,-0.119,-0.005,0.000,0.0,0.798,0.0837,0.1402,0.7536,0.0000,0.1369,0.0016,0.0848,0.0543,0.8031
4,6,Saliba,William,Saliba,DEF,Arsenal,6.2,37,361,Burnley,True,84.1,30,100.0,4.76,1.998,2.497,-0.234,0.306,0.069,0.000,-0.080,-0.005,0.000,0.0,0.204,0.0337,0.0152,0.7786,0.0000,0.0869,0.0016,0.0333,0.0416,0.7863
5,6,Saliba,William,Saliba,DEF,Arsenal,6.2,38,373,Crystal Palace,False,84.1,30,100.0,3.20,1.998,1.408,-0.521,0.160,0.036,0.000,-0.080,-0.005,0.000,0.0,0.204,0.0337,0.0152,0.7786,0.0000,0.0869,0.0016,0.0333,0.0416,0.7863
6,7,Calafiori,Riccardo,Calafiori,DEF,Arsenal,5.6,37,361,Burnley,True,65.9,24,75.0,3.34,1.783,2.231,-0.169,0.360,0.349,0.000,-0.157,-0.006,0.000,0.0,0.055,0.0396,0.0768,0.5630,0.0000,0.1904,0.0019,0.1121,0.0292,0.6092
7,7,Calafiori,Riccardo,Calafiori,DEF,Arsenal,5.6,38,373,Crystal Palace,False,65.9,24,75.0,3.14,1.783,1.474,-0.377,0.188,0.182,0.000,-0.157,-0.006,0.000,0.0,0.055,0.0396,0.0768,0.5630,0.0000,0.1904,0.0019,0.1121,0.0292,0.6092
8,8,J.Timber,Jurriën,Timber,DEF,Arsenal,6.0,37,361,Burnley,True,81.7,30,0.0,0.00,1.995,2.491,-0.234,0.761,0.751,0.000,-0.138,-0.005,0.000,0.0,0.105,0.0837,0.1652,0.7786,0.0000,0.1619,0.0016,0.1291,0.0498,0.7171
9,8,J.Timber,Jurriën,Timber,DEF,Arsenal,6.0,38,373,Crystal Palace,False,81.7,30,0.0,3.63,1.995,1.404,-0.521,0.397,0.392,0.000,-0.138,-0.005,0.000,0.0,0.105,0.0837,0.1652,0.7786,0.0000,0.1619,0.0016,0.1291,0.0498,0.7171


---
## Section 8 — Player Summary DataFrame

One row per player. Aggregates everything into the final table:
identity, season stats, per-90 stats, FPL metrics, model output, future projections.

In [28]:
def build_player_summary(
    players_raw: pd.DataFrame,
    performances: pd.DataFrame,
    future_predictions: pd.DataFrame,
    player_rates: pd.DataFrame,
    next_gw: int,
    n_gws: int = 5
) -> pd.DataFrame:
    """
    Build the Player Summary DataFrame.

    GW xPts columns:
        - DGW: sum of two fixture xPts for that week
        - BGW: 0 (filled explicitly — no fixture)
    """
    gw_range = range(next_gw, next_gw + n_gws)

    # ── Season totals ─────────────────────────────────────────────────────────
    totals = performances.groupby('player_id').agg(
        MP=('minutes', lambda x: (x > 0).sum()),
        total_minutes=('minutes', 'sum'),
        goals=('goals_scored', 'sum'),
        assists=('assists', 'sum'),
        clean_sheets=('clean_sheets', 'sum'),
        goals_conceded=('goals_conceded', 'sum'),
        saves=('saves', 'sum'),
        yellow_cards=('yellow_cards', 'sum'),
        red_cards=('red_cards', 'sum'),
        bonus=('bonus', 'sum'),
        bps_total=('bps', 'sum'),
        total_xG=('xG', 'sum'),
        total_xA=('xA', 'sum'),
        total_xGC=('xGC', 'sum'),
        total_points=('total_points', 'sum'),
    ).reset_index()

    # Per-90
    nineties = totals['total_minutes'] / 90
    totals['90s']         = nineties.round(1)
    totals['goals_p90']   = (totals['goals']    / nineties).round(3)
    totals['assists_p90'] = (totals['assists']   / nineties).round(3)
    totals['xG_p90']      = (totals['total_xG'] / nineties).round(3)
    totals['xA_p90']      = (totals['total_xA'] / nineties).round(3)
    totals['xGC_p90']     = (totals['total_xGC']/ nineties).round(3)
    totals['pts_p90']     = (totals['total_points'] / nineties).round(3)

    # ── Latest form ───────────────────────────────────────────────────────────
    form = (
        performances.sort_values('gameweek')
        .groupby('player_id')[['xForm5', 'xForm10']]
        .last()
        .reset_index()
    )

    # ── Baseline xPts (no opponent adjustment) ────────────────────────────────
    baseline = player_rates.copy()
    baseline['xPts_baseline'] = baseline.apply(
        lambda r: compute_xpts(r, mode='projected')['xPts'], axis=1
    )

    # ── Per-GW xPts projections (sum across fixtures per GW) ─────────────────
    gw_xpts = (
        future_predictions.groupby(['id', 'gameweek'])['xPts']
        .sum()
        .reset_index()
        .rename(columns={'id': 'player_id'})
    )
    gw_pivot = gw_xpts.pivot(
        index='player_id', columns='gameweek', values='xPts'
    ).reset_index()
    gw_pivot.columns.name = None
    gw_pivot.columns = [
        'player_id' if c == 'player_id' else f'xPts_GW{c}'
        for c in gw_pivot.columns
    ]

    # Ensure all GW columns exist; fill BGW (blank) with 0
    gw_cols = [f'xPts_GW{gw}' for gw in gw_range]
    for col in gw_cols:
        if col not in gw_pivot.columns:
            gw_pivot[col] = 0.0
        else:
            gw_pivot[col] = gw_pivot[col].fillna(0.0).round(2)

    gw_pivot['xPts_next5_sum'] = gw_pivot[gw_cols].sum(axis=1).round(2)
    gw_pivot['xPts_next5_avg'] = gw_pivot[gw_cols].mean(axis=1).round(2)

    # ── Base player info ──────────────────────────────────────────────────────
    base = players_raw[[
        'id', 'first_name', 'second_name', 'web_name',
        'position', 'team', 'team_name', 'price_m',
        'selected_by_percent',
        'chance_of_playing_this_round', 'chance_of_playing_next_round',
        'form', 'points_per_game'
    ]].copy()
    base['chance_of_playing_this_round'] = base['chance_of_playing_this_round'].fillna(100.0)
    base['chance_of_playing_next_round'] = base['chance_of_playing_next_round'].fillna(100.0)
    base['points_per_game'] = pd.to_numeric(base['points_per_game'], errors='coerce')

    # ── Assemble ──────────────────────────────────────────────────────────────
    summary = (
        base
        .merge(totals,                                       left_on='id', right_on='player_id', how='left')
        .merge(form,                                         left_on='id', right_on='player_id', how='left')
        .merge(baseline[['xPts_baseline']],                  left_on='id', right_index=True,    how='left')
        .merge(gw_pivot,                                     left_on='id', right_on='player_id', how='left')
    )
    summary.drop(columns=[c for c in summary.columns if c == 'player_id'], inplace=True)

    # ── Value metrics ─────────────────────────────────────────────────────────
    # xVAPM: expected value added per million vs. minimum-bench-player baseline (2 pts)
    summary['xVAPM'] = ((summary[f'xPts_GW{next_gw}'] - 2) / summary['price_m']).round(3)
    summary['VAPM']  = ((summary['points_per_game']    - 2) / summary['price_m']).round(3)

    # ── Final column order ────────────────────────────────────────────────────
    id_cols   = ['id', 'first_name', 'second_name', 'web_name', 'position',
                 'team_name', 'price_m', 'selected_by_percent',
                 'chance_of_playing_this_round', 'chance_of_playing_next_round']
    stat_cols = ['MP', '90s', 'total_minutes', 'goals', 'assists', 'clean_sheets',
                 'goals_conceded', 'saves', 'yellow_cards', 'red_cards',
                 'bonus', 'bps_total', 'total_xG', 'total_xA', 'total_xGC', 'total_points']
    p90_cols  = ['goals_p90', 'assists_p90', 'xG_p90', 'xA_p90', 'xGC_p90', 'pts_p90']
    fpl_cols  = ['form', 'points_per_game']
    mdl_cols  = ['xForm5', 'xForm10', 'xPts_baseline', 'xVAPM', 'VAPM']
    proj_cols = gw_cols + ['xPts_next5_sum', 'xPts_next5_avg']

    all_cols = id_cols + stat_cols + p90_cols + fpl_cols + mdl_cols + proj_cols
    summary  = summary[[c for c in all_cols if c in summary.columns]].round(2)

    log.info(f'Player Summary built: {summary.shape}')
    return summary


player_summary = build_player_summary(
    players_raw, performances, future_predictions, player_rates, next_gw
)

player_summary.to_excel(
    os.path.join(PROJECT_DIR, f'fpl_xpts_GW{next_gw}.xlsx'),
    sheet_name='Players', index=False
)
player_summary.to_csv(
    os.path.join(PROJECT_DIR, 'player_summary.csv'), index=False
)

print(f'Player Summary: {player_summary.shape}')
player_summary.sort_values('xPts_next5_sum', ascending=False).head(20)

2026-05-14 15:40:23,745  INFO      Player Summary built: (490, 46)


Player Summary: (490, 46)


,id,first_name,second_name,web_name,position,team_name,price_m,selected_by_percent,chance_of_playing_this_round,chance_of_playing_next_round,MP,90s,total_minutes,goals,assists,clean_sheets,goals_conceded,saves,yellow_cards,red_cards,bonus,bps_total,total_xG,total_xA,total_xGC,total_points,goals_p90,assists_p90,xG_p90,xA_p90,xGC_p90,pts_p90,form,points_per_game,xForm5,xForm10,xPts_baseline,xVAPM,VAPM,xPts_GW37,xPts_GW38,xPts_GW39,xPts_GW40,xPts_GW41,xPts_next5_sum,xPts_next5_avg
1,5,Gabriel,dos Santos Magalhães,Gabriel,DEF,Arsenal,7.3,45.6,100.0,100.0,30,29.1,2615,3,5,17,19,0,4,0,30,695,2.94,1.71,20.89,202,0.10,0.17,0.10,0.06,0.72,6.95,6.2,6.7,5.12,5.19,5.10,0.60,0.64,6.38,4.36,0.0,0.0,0.0,10.74,2.15
329,449,Bruno,Borges Fernandes,B.Fernandes,MID,Man Utd,10.4,47.6,100.0,100.0,33,32.1,2885,8,21,6,44,0,5,0,37,916,10.53,11.25,41.75,212,0.25,0.66,0.33,0.35,1.30,6.61,4.8,6.4,3.82,4.97,4.85,0.32,0.42,5.36,4.83,0.0,0.0,0.0,10.19,2.04
204,291,James,Tarkowski,Tarkowski,DEF,Everton,5.7,11.9,100.0,100.0,35,35.0,3150,2,3,11,44,0,7,0,12,583,2.40,2.08,51.63,164,0.06,0.09,0.07,0.06,1.48,4.69,4.8,4.7,3.12,3.69,3.86,0.57,0.47,5.25,4.39,0.0,0.0,0.0,9.64,1.93
50,183,Max,Weiß,Weiss,GKP,Burnley,4.2,0.1,100.0,100.0,1,1.0,90,0,0,0,2,5,0,0,0,13,0.00,0.00,1.44,2,0.00,0.00,0.00,0.00,1.44,2.00,0.5,2.0,3.61,3.61,3.26,0.19,0.00,2.78,6.86,0.0,0.0,0.0,9.64,1.93
71,470,Martin,Dúbravka,Dúbravka,GKP,Burnley,4.0,26.3,100.0,100.0,35,35.0,3150,0,0,4,71,127,1,0,7,436,0.00,0.04,71.26,96,0.00,0.00,0.00,0.00,2.04,2.74,1.5,2.7,2.53,2.42,2.80,0.08,0.18,2.34,6.94,0.0,0.0,0.0,9.28,1.86
208,295,Michael,Keane,Keane,DEF,Everton,4.5,2.7,100.0,100.0,31,26.8,2408,3,0,7,38,0,1,1,8,386,2.21,0.23,42.61,123,0.11,0.00,0.08,0.01,1.59,4.60,1.2,4.0,2.71,2.41,3.66,0.67,0.44,5.02,4.19,0.0,0.0,0.0,9.21,1.84
151,225,Reece,James,James,DEF,Chelsea,5.6,5.9,100.0,100.0,28,21.3,1919,2,6,8,20,0,4,0,11,488,1.02,3.16,28.29,114,0.09,0.28,0.05,0.15,1.33,5.35,0.2,4.1,3.46,2.96,3.80,0.51,0.38,4.84,4.29,0.0,0.0,0.0,9.13,1.83
152,226,Trevoh,Chalobah,Chalobah,DEF,Chelsea,5.4,9.0,100.0,100.0,32,30.4,2739,3,1,9,40,0,3,1,7,507,1.44,1.03,40.34,134,0.10,0.03,0.05,0.03,1.33,4.40,0.8,4.2,2.25,3.00,3.52,0.52,0.41,4.78,4.32,0.0,0.0,0.0,9.10,1.82
357,488,Bruno,Guimarães Rodriguez Moura,Bruno G.,MID,Newcastle,6.8,5.5,100.0,100.0,27,25.6,2301,9,7,6,35,0,5,0,22,657,5.48,4.55,29.82,149,0.35,0.27,0.21,0.18,1.17,5.83,3.8,5.5,4.16,5.18,4.38,0.45,0.52,5.06,3.98,0.0,0.0,0.0,9.04,1.81
52,190,Hjalmar,Ekdal,Ekdal,DEF,Burnley,3.8,0.7,100.0,100.0,19,17.1,1539,0,0,2,34,0,0,0,2,159,0.25,0.32,31.03,52,0.00,0.00,0.02,0.02,1.82,3.04,1.8,2.7,2.71,2.83,2.92,0.09,0.18,2.33,6.68,0.0,0.0,0.0,9.01,1.80


---
## Section 9 — Inspection Helpers

Utility functions for spot-checking model output during development.

In [30]:
def inspect_player(name: str):
    """
    Print a full breakdown of xPts components and future predictions for one player.
    Accepts any part of the player's name (web_name, first or last name).

    Usage:
        inspect_player('Salah')
        inspect_player('Haaland')
    """
    mask = (
        player_summary['web_name'].str.contains(name, case=False, na=False) |
        player_summary['first_name'].str.contains(name, case=False, na=False) |
        player_summary['second_name'].str.contains(name, case=False, na=False)
    )
    matches = player_summary[mask]

    if matches.empty:
        print(f'No player found matching "{name}".')
        return

    for _, row in matches.iterrows():
        pid = row['id']
        print(f"{'─'*60}")
        print(f"  {row['web_name']} ({row['position']}) — {row['team_name']} — £{row['price_m']}m")
        print(f"  MP: {row['MP']} | 90s: {row['90s']} | Avail: {row['chance_of_playing_next_round']}%")
        print(f"  xForm5: {row['xForm5']} | xForm10: {row['xForm10']}")
        print(f"  xPts baseline: {row['xPts_baseline']} | xVAPM: {row['xVAPM']} | VAPM: {row['VAPM']}")
        print()

        gw_cols = [c for c in player_summary.columns if c.startswith('xPts_GW')]
        print('  Future GW projections:')
        for col in gw_cols:
            gw_num = int(col.replace('xPts_GW', ''))
            fx     = future_predictions[
                (future_predictions['id'] == pid) &
                (future_predictions['gameweek'] == gw_num)
            ]
            if fx.empty:
                print(f'    GW{gw_num}: BLANK — 0.0 xPts')
            else:
                for _, frow in fx.iterrows():
                    ha = 'H' if frow['is_home'] else 'A'
                    print(f'    GW{gw_num}: vs {frow["opponent_team"]} ({ha}) → {frow["xPts"]} xPts')
                if len(fx) > 1:
                    print(f'           DGW total → {row[col]} xPts')

        print(f"  Next-5 sum: {row['xPts_next5_sum']} | avg: {row['xPts_next5_avg']}")


def top_picks(position: str = None, n: int = 15, sort_by: str = 'xPts_next5_sum'):
    """
    Show top n players ranked by sort_by.

    Parameters
    ----------
    position : 'GKP' | 'DEF' | 'MID' | 'FWD' | None (all positions)
    n        : number of players to show
    sort_by  : any column in player_summary (default: 'xPts_next5_sum')

    Usage:
        top_picks('MID')
        top_picks('FWD', sort_by='xVAPM')
        top_picks(sort_by='xForm5')
    """
    df = player_summary.copy()
    if position:
        df = df[df['position'] == position.upper()]

    gw_cols  = [c for c in df.columns if c.startswith('xPts_GW')]
    show_cols = [
        'web_name', 'position', 'team_name', 'price_m',
        'xForm5', 'xPts_baseline', 'xVAPM'
    ] + gw_cols + ['xPts_next5_sum']

    return df.sort_values(sort_by, ascending=False)[show_cols].head(n)


print('Helpers ready.')
print("  inspect_player('Salah')")
print("  top_picks('MID')")
print("  top_picks('FWD', sort_by='xVAPM')")

Helpers ready.
  inspect_player('Salah')
  top_picks('MID')
  top_picks('FWD', sort_by='xVAPM')


---
## Section 10 – Exporting to Excel

To see all players in one file.

In [32]:
with pd.ExcelWriter(f'fpl_projections_asof{next_gw}.xlsx') as writer:
    player_summary.to_excel(writer, sheet_name="Player Summary", index=False)
    future_predictions.to_excel(writer, sheet_name="Future Predictions", index=False)
    performances.to_excel(writer, sheet_name="Performances", index=False)